<a href="https://colab.research.google.com/github/DifferentiableUniverseInitiative/JaxPM/blob/main/docs/notebooks/09-Multigrid_Poisson_Solver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multigrid Poisson solver

`pm_forces` solves the Poisson equation with an FFT by default. With `solver="mg"` it uses
a real-space geometric multigrid solver instead, following
[Horowitz (2026)](https://arxiv.org/abs/2607.10983): V-cycles on the discrete 7-point
Laplacian, followed by a 4th-order finite-difference gradient. No Fourier transform is
involved, so in a distributed run the only communication is a halo exchange between
neighbouring devices, instead of the all-to-all transposes of a distributed FFT.

Multigrid is iterative, so it can also start from an initial guess `u0`. The potential changes
slowly between PM steps, and seeding the solve with the previous potential reduces the number
of V-cycles needed for a given accuracy. This notebook runs on a single GPU and covers:

1. FFT and multigrid forces on the same density field;
2. convergence with the number of V-cycles, from a cold and a warm start;
3. a full PM simulation with each solver, compared with the halofit power spectrum.

In [ ]:
# Install JaxPM (multigrid branch) and diffrax. Runs only when the packages are
# missing (e.g. on Colab); an existing local install is left untouched.
try:
    import jaxpm  # noqa: F401
    import diffrax  # noqa: F401
except ImportError:
    %pip install -q "jaxpm @ git+https://github.com/DifferentiableUniverseInitiative/JaxPM.git@bhorowitz/MG"
    %pip install -q diffrax

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import jax_cosmo as jc
import matplotlib.pyplot as plt
import numpy as np
from diffrax import ConstantStepSize, ODETerm, SaveAt, SemiImplicitEuler, diffeqsolve

from jaxpm.growth import growth_factor
from jaxpm.ode import symplectic_ode
from jaxpm.painting import paint
from jaxpm.pm import linear_field, lpt, pm_forces
from jaxpm.utils import power_spectrum

print(jax.devices())

## Initial conditions

A $256^3$ Gaussian field in a $(256\,h^{-1}\mathrm{Mpc})^3$ box. First-order LPT gives the
particle displacements at the start of the simulation, $a_0 = 0.1$, and at $a = 1$. The
$a = 1$ displacements produce a strongly clustered density, which is the harder case for the
force comparison below.

In [ ]:
mesh_shape = (256, 256, 256)
box_size = [256., 256., 256.]
a0 = 0.1

cosmo = jc.Planck15(Omega_c=0.25, sigma8=0.8)
k = jnp.logspace(-4, 1, 128)
pk = jc.power.linear_matter_power(cosmo, k)
pk_fn = lambda x: jnp.interp(x.reshape([-1]), k, pk).reshape(x.shape)

initial_conditions = linear_field(mesh_shape, box_size, pk_fn, seed=jax.random.PRNGKey(0))

dx0, p0, _ = lpt(cosmo, initial_conditions, a=a0, order=1)
dx1, _, _ = lpt(cosmo, initial_conditions, a=1.0, order=1)
print(dx0.shape, dx1.shape)

## FFT and multigrid forces on the same density

The particles are given as displacements from a uniform grid (`initial_particles="uniform"`).
The multigrid solver discretises the Laplacian with finite differences, which corresponds to
the FFT solver with `laplace_fd=True`. Comparing against that reference isolates the
multigrid convergence error. The default FFT solver (`laplace_fd=False`) inverts the exact
$k^2$ and differs from both by the discretisation error of the Laplacian.

In [ ]:
f_fft = pm_forces(dx1, mesh_shape=mesh_shape, initial_particles="uniform")
f_fft_fd = pm_forces(dx1, mesh_shape=mesh_shape, initial_particles="uniform", laplace_fd=True)
f_mg = pm_forces(dx1, mesh_shape=mesh_shape, initial_particles="uniform",
                 solver="mg", mg_params={"cycles": 4})

norm = jnp.linalg.norm
print(f"MG (4 cycles) vs FFT laplace_fd=True   : {norm(f_mg - f_fft_fd) / norm(f_fft_fd):.2e}")
print(f"MG (4 cycles) vs FFT laplace_fd=False  : {norm(f_mg - f_fft) / norm(f_fft):.2e}")
print(f"FFT laplace_fd=True vs laplace_fd=False: {norm(f_fft_fd - f_fft) / norm(f_fft):.2e}")

In [ ]:
z = mesh_shape[2] // 2
fx_fft = np.asarray(f_fft_fd[:, :, z, 0])
fx_mg = np.asarray(f_mg[:, :, z, 0])
vmax = np.abs(fx_fft).max()
dmax = np.abs(fx_mg - fx_fft).max()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
im = axes[0].imshow(fx_fft, cmap="magma", vmin=-vmax, vmax=vmax)
fig.colorbar(im, ax=axes[0], fraction=0.046)
axes[0].set_title("FFT (laplace_fd=True)")
im = axes[1].imshow(fx_mg, cmap="magma", vmin=-vmax, vmax=vmax)
fig.colorbar(im, ax=axes[1], fraction=0.046)
axes[1].set_title("Multigrid (4 cycles)")
im = axes[2].imshow(fx_mg - fx_fft, cmap="RdBu_r", vmin=-dmax, vmax=dmax)
fig.colorbar(im, ax=axes[2], fraction=0.046)
axes[2].set_title("Multigrid - FFT")
for ax in axes:
    ax.axis("off")
fig.suptitle("x-component of the force, slice z = N/2")
plt.tight_layout()
plt.show()

## Convergence with the number of V-cycles

A relaxation sweep (here weighted Jacobi) removes the error components that oscillate on the
scale of the grid within a few iterations, but it is very slow on the smooth, large-scale
components. Multigrid handles those on coarser grids. One V-cycle:

1. smooths the current solution with `v1` sweeps on the fine grid;
2. restricts the residual $f - \nabla^2 \phi$ to a grid twice as coarse;
3. repeats steps 1 and 2 recursively down to the coarsest level, where the problem is small;
4. on the way back up, interpolates each coarse correction to the finer grid and smooths
   with `v2` sweeps.

Each level removes the error on its own scale, so one V-cycle reduces the error by a roughly
constant factor, independent of the mesh size, for $O(N^3)$ work. `mg_params` sets `cycles`
(the number of V-cycles), `v1`, `v2`, the relaxation weight `omega`, and `levels` (inferred
from the mesh shape by default).

A cold start (`u0=None`) begins from a zero potential.

In [ ]:
cycle_list = [1, 2, 3, 4, 6]
cold_err = []
for cycles in cycle_list:
    f = pm_forces(dx1, mesh_shape=mesh_shape, initial_particles="uniform",
                  solver="mg", mg_params={"cycles": cycles})
    cold_err.append(norm(f - f_fft_fd) / norm(f_fft_fd))
    print(f"cycles={cycles}: relative force error {cold_err[-1]:.2e}")

## Warm start

In a simulation the potential of the previous step is already known. Here the previous
step is $a = 0.9$: its potential, solved to convergence and rescaled by the linear growth
$D(1)/D(0.9)$, is passed as `u0` for the solve at $a = 1$. `return_potential=True` makes
`pm_forces` return the potential along with the forces.

In [ ]:
a_prev = 0.9
dx_prev, _, _ = lpt(cosmo, initial_conditions, a=a_prev, order=1)
_, phi_prev = pm_forces(dx_prev, mesh_shape=mesh_shape, initial_particles="uniform",
                        solver="mg", mg_params={"cycles": 6}, return_potential=True)

growth = growth_factor(cosmo, jnp.atleast_1d(1.0)) / growth_factor(cosmo, jnp.atleast_1d(a_prev))
u0 = phi_prev * growth

warm_err = []
for cycles, cold in zip(cycle_list, cold_err):
    f = pm_forces(dx1, mesh_shape=mesh_shape, initial_particles="uniform",
                  solver="mg", mg_params={"cycles": cycles}, u0=u0)
    warm_err.append(norm(f - f_fft_fd) / norm(f_fft_fd))
    print(f"cycles={cycles}: cold {cold:.2e}  warm {warm_err[-1]:.2e}")

In [ ]:
plt.figure(figsize=(5, 4))
plt.semilogy(cycle_list, cold_err, "o-", label="cold start")
plt.semilogy(cycle_list, warm_err, "s-", label="warm start")
plt.xlabel("V-cycles")
plt.ylabel("relative force error vs FFT (laplace_fd=True)")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

## PM simulation

The same initial conditions are evolved from $a_0 = 0.1$ to $a = 1$ in 18 steps with
`symplectic_ode` and a semi-implicit Euler integrator, once with each solver. `symplectic_ode`
passes `solver` and `mg_params` to `pm_forces`; each step starts the multigrid solve from
zero, because the ODE right-hand side does not carry the previous potential. The FFT run
uses the default exact-$k^2$ kernel, so the comparison includes the Laplacian discretisation
error as well as the multigrid convergence error.

In [ ]:
final_density = {}
for name, solver, mg_params in [("FFT", "fft", None), ("Multigrid", "mg", {"cycles": 4})]:
    drift, kick = symplectic_ode(mesh_shape, cosmo, initial_particles="uniform",
                                 solver=solver, mg_params=mg_params)
    sol = diffeqsolve(
        (ODETerm(kick), ODETerm(drift)),
        SemiImplicitEuler(),
        t0=a0,
        t1=1.0,
        dt0=0.05,
        y0=(p0, dx0),
        args=cosmo,
        saveat=SaveAt(t1=True),
        stepsize_controller=ConstantStepSize(),
    )
    final_density[name] = paint(sol.ys[1][-1], initial_particles="uniform")

In [ ]:
proj_fft = np.asarray(final_density["FFT"][:, :, :16].sum(axis=-1))
proj_mg = np.asarray(final_density["Multigrid"][:, :, :16].sum(axis=-1))
diff = proj_mg - proj_fft
dlim = np.percentile(np.abs(diff), 99)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(np.log10(proj_fft + 1), cmap="magma")
axes[0].set_title("FFT")
axes[1].imshow(np.log10(proj_mg + 1), cmap="magma")
axes[1].set_title("Multigrid (4 cycles)")
axes[2].imshow(diff, cmap="RdBu_r", vmin=-dlim, vmax=dlim)
axes[2].set_title("Multigrid - FFT")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

The power spectra of both runs are compared with the linear and halofit predictions from
`jax_cosmo`. A PM code with one particle per cell and 18 steps is expected to lose power
relative to halofit on small scales; the two solvers should agree with each other.

In [ ]:
k_fft, pk_fft = power_spectrum(final_density["FFT"], box_shape=box_size)
k_mg, pk_mg = power_spectrum(final_density["Multigrid"], box_shape=box_size)
pk_linear = jc.power.linear_matter_power(cosmo, k_fft, a=1.0)
pk_halofit = jc.power.nonlinear_matter_power(cosmo, k_fft, a=1.0)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4))
ax0.loglog(k_fft, pk_linear, "k:", label="linear")
ax0.loglog(k_fft, pk_halofit, "k--", label="halofit")
ax0.loglog(k_fft, pk_fft, label="FFT")
ax0.loglog(k_mg, pk_mg, label="Multigrid")
ax0.set_xlabel(r"$k$ [$h$/Mpc]")
ax0.set_ylabel(r"$P(k)$ [(Mpc/$h$)$^3$]")
ax0.legend()

ax1.semilogx(k_fft, pk_fft / pk_halofit, label="FFT / halofit")
ax1.semilogx(k_mg, pk_mg / pk_halofit, label="Multigrid / halofit")
ax1.semilogx(k_mg, pk_mg / pk_fft, label="Multigrid / FFT")
ax1.axhline(1, color="k", lw=0.8)
ax1.set_xlabel(r"$k$ [$h$/Mpc]")
ax1.set_ylabel("ratio")
ax1.legend()
plt.tight_layout()
plt.show()